In [ ]:
from pathlib import Path
import os
import numpy as np
import arviz as az
import pickle
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls

from emu_renewal.inputs import OUTPUTS_PATH
from emu_renewal.outputs import load_targets
from emu_renewal.plotting import plot_multianalysis_fit, plot_post_prior_comparison

set_matplotlib_formats("svg")

In [ ]:
import pycountry
pycountry.countries.lookup("BGD")

In [ ]:
job_path = OUTPUTS_PATH / "44044421"
countries = [c for c in ls(job_path) if c != "AUS"]
n_cols = 3
for c in ["BGD"]:
    print(c)
    analysis_dict = {d.name: Path(d.path) for d in os.scandir(job_path / c) if d.is_dir()}

    if "no_mob" in analysis_dict:
        spaghs = {a: pd.read_hdf(p / "spaghetti.h5") for a, p in analysis_dict.items()}
        no_mob_path = analysis_dict["no_mob"]
        targs = load_targets(no_mob_path)
        plot_multianalysis_fit(c, targs, spaghs)
        
        idata = az.from_netcdf(no_mob_path / "idata_filtered.nc");
        priors = pickle.load(open(no_mob_path / "priors.pkl", "rb"))
        epi_params = [p for p in priors if p != "shared_dispersion"]
        n_rows = int(np.ceil(len(priors) / n_cols)) + 1
        plot_post_prior_comparison(idata, epi_params, priors, c, req_grid=[n_rows, n_cols], req_size=[12, 15]);
    else:
        print(f"no_mob not available for country: {c}")